# 02 — Multi-feature regression

## Goal
Extend the single-feature baseline from `01_baseline_regression.ipynb` to a multi-feature linear regression model.

This notebook also covers feature scaling and feature engineering.

## Dataset
Starts from the train/test split produced in `00_data_cleaning.ipynb`.

## Target
`rentGross`, same choice and reasoning as in `00_data_cleaning.ipynb`.

----------

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import math

In [2]:
train_df = pd.read_csv('../data/train.csv')
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 586 entries, 0 to 585
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   type           586 non-null    str    
 1   lat            586 non-null    float64
 2   lon            586 non-null    float64
 3   livingSpace    586 non-null    float64
 4   numberOfRooms  586 non-null    float64
 5   floor          586 non-null    float64
 6   hasGarage      586 non-null    str    
 7   hasParking     586 non-null    str    
 8   hasBalcony     586 non-null    str    
 9   hasElevator    586 non-null    str    
 10  rentGross      586 non-null    float64
dtypes: float64(6), str(5)
memory usage: 50.5 KB


In [3]:
train_df.head()

,type,lat,lon,livingSpace,numberOfRooms,floor,hasGarage,hasParking,hasBalcony,hasElevator,rentGross
0,FLAT,47.397272,8.601042,147.0,4.5,26.0,unknown,unknown,True,True,4950.0
1,LOFT,47.213072,7.789612,167.0,2.5,2.0,True,unknown,unknown,unknown,2225.0
2,FLAT,47.344682,8.734462,57.0,2.5,3.0,True,unknown,True,unknown,1782.0
3,FLAT,47.661842,8.978512,103.0,3.5,2.0,unknown,True,True,unknown,1600.0
4,FLAT,47.521512,8.930522,97.0,4.5,1.0,unknown,True,unknown,True,1985.0


## 1. Encode categorical features

Two categorical columns need encoding: `type`, and the four `has*` columns (`hasGarage`, `hasParking`, `hasBalcony`, `hasElevator`).

The `has*` columns aren't boolean despite the name. We keep `unknown` and `True` as explicit categories in the one-hot encoding, with `False` as the reference category since it's the rarest value across all four columns.

`type` gets a standard one-hot encoding.

In [4]:
type_counts = train_df['type'].value_counts()
common_types = type_counts[type_counts >= 10].index

# Keep the value as is if it is a common type, replace with "other" otherwise.
train_df['type'] = train_df['type'].where(train_df['type'].isin(common_types), 'other')
train_df['type'].value_counts()

type
FLAT            463
ROOF_FLAT        32
DUPLEX           29
other            28
ATTIC_FLAT       24
TERRACE_FLAT     10
Name: count, dtype: int64

In [5]:
train_df = pd.get_dummies(train_df, columns=['type', 'hasGarage', 'hasParking', 'hasBalcony', 'hasElevator'], drop_first=True, dtype=int)
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 586 entries, 0 to 585
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   lat                  586 non-null    float64
 1   lon                  586 non-null    float64
 2   livingSpace          586 non-null    float64
 3   numberOfRooms        586 non-null    float64
 4   floor                586 non-null    float64
 5   rentGross            586 non-null    float64
 6   type_DUPLEX          586 non-null    int64  
 7   type_FLAT            586 non-null    int64  
 8   type_ROOF_FLAT       586 non-null    int64  
 9   type_TERRACE_FLAT    586 non-null    int64  
 10  type_other           586 non-null    int64  
 11  hasGarage_True       586 non-null    int64  
 12  hasGarage_unknown    586 non-null    int64  
 13  hasParking_True      586 non-null    int64  
 14  hasParking_unknown   586 non-null    int64  
 15  hasBalcony_True      586 non-null    int64  
 16  h

Let's encode the test set the same way, and verify the columns match between the two datasets.

In [6]:
test_df = pd.read_csv('../data/test.csv')
test_df['type'] = test_df['type'].where(test_df['type'].isin(common_types), 'other')
test_df['type'].value_counts()

type
FLAT            117
ATTIC_FLAT        9
DUPLEX            9
ROOF_FLAT         6
other             4
TERRACE_FLAT      2
Name: count, dtype: int64

In [7]:
test_df = pd.get_dummies(test_df, columns=['type', 'hasGarage', 'hasParking', 'hasBalcony', 'hasElevator'], drop_first=True, dtype=int)
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 147 entries, 0 to 146
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   lat                  147 non-null    float64
 1   lon                  147 non-null    float64
 2   livingSpace          147 non-null    float64
 3   numberOfRooms        147 non-null    float64
 4   floor                147 non-null    float64
 5   rentGross            147 non-null    float64
 6   type_DUPLEX          147 non-null    int64  
 7   type_FLAT            147 non-null    int64  
 8   type_ROOF_FLAT       147 non-null    int64  
 9   type_TERRACE_FLAT    147 non-null    int64  
 10  type_other           147 non-null    int64  
 11  hasGarage_True       147 non-null    int64  
 12  hasGarage_unknown    147 non-null    int64  
 13  hasParking_True      147 non-null    int64  
 14  hasParking_unknown   147 non-null    int64  
 15  hasBalcony_True      147 non-null    int64  
 16  h

In [8]:
print(set(train_df.columns) - set(test_df.columns))
print(set(test_df.columns) - set(train_df.columns))

set()
set()


Both sets are empty, train and test have identical columns after encoding, confirming the two datasets are structurally aligned for training.

## 2. Feature engineering

Two new features are built from the columns already available.

- **Distance to nearest population center**: replaces `lat`/`lon` as the geographic signal.
- **Average room size**: `livingSpace / numberOfRooms`. Two listings with the same total square meters can be split very differently between rooms.  `livingSpace` and `numberOfRooms` don't capture this information on their own when used separately.

## 2.1 Distance to nearest population center

For each listing, we compute the distance to every city in a reference list of Swiss population centers, and keep the distance to the closest one.

**City list dataset**: Swiss municipalities with population ≥ 10,000, filtered from the [GeoNames Switzerland dump](https://download.geonames.org/export/dump/CH.zip). This gives 164 reference points spread across the country, not just the well-known cities.

In [9]:
cities_df = pd.read_csv('../data/CH.txt', sep='\t', header=None)
cities_df.head()

/var/folders/w6/7cm71xss70n37td3zgkt_5xh0000gn/T/ipykernel_36773/3227567970.py:1: DtypeWarning: Columns (0: 9) have mixed types. Specify dtype option on import or set low_memory=False.
  cities_df = pd.read_csv('../data/CH.txt', sep='\t', header=None)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,2657883,Zuger See,Zuger See,"Lac de Zoug,Lago di Zug,Lai da Zug,Lake Zug,La...",47.13130,8.48335,H,LK,CH,NaN,00,NaN,NaN,NaN,0,413.0,411,Europe/Zurich,2012-02-01
1,2657884,Zwischbergental,Zwischbergental,"Zwischberg-Thal,Zwischbergental",46.16667,8.13333,T,VAL,CH,NaN,VS,NaN,NaN,NaN,0,NaN,1671,Europe/Zurich,2012-01-17
2,2657885,Zwischbergen,Zwischbergen,"Zwischbergen,ci wei shi bei gen,茨維施貝根",46.16366,8.11575,P,PPL,CH,NaN,VS,2301.0,6011.0,NaN,127,NaN,1322,Europe/Zurich,2012-01-17
3,2657886,Zwingen,Zwingen,"Cvingen,ci wen gen,Цвинген,茨溫根",47.43825,7.53027,P,PPL,CH,NaN,BL,1302.0,2793.0,NaN,2162,NaN,342,Europe/Zurich,2013-02-28
4,2657887,Zweisimmen,Zweisimmen,"Cvajzimmen,Zweisimmen,Zweisimmeni vald,ci wei ...",46.55539,7.37302,P,PPL,CH,NaN,BE,248.0,794.0,NaN,2813,NaN,945,Europe/Zurich,2017-02-03


In [10]:
cities_df = cities_df.rename(columns={1: 'name', 4: 'latitude', 5: 'longitude', 6: 'feature_class', 14: 'population'})
cities_df = cities_df[(cities_df['feature_class'] == 'P') & (cities_df['population'] >= 10000)]
cities_df = cities_df[['name', 'latitude', 'longitude']]
cities_df.info()

<class 'pandas.DataFrame'>
Index: 164 entries, 13 to 11270
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       164 non-null    str    
 1   latitude   164 non-null    float64
 2   longitude  164 non-null    float64
dtypes: float64(2), str(1)
memory usage: 5.1 KB


In [11]:
cities_df.head()

,name,latitude,longitude
13,Zürich,47.36667,8.55000
24,Zug,47.17242,8.51745
31,Zofingen,47.28779,7.94586
56,Yverdon-les-Bains,46.77852,6.64115
68,Worb,46.92984,7.56306


In [12]:
lat_cities = cities_df['latitude'].values
lon_cities = cities_df['longitude'].values

def min_distance(row):
    distances = np.sqrt((row['lat'] - lat_cities)**2 + (row['lon'] - lon_cities)**2)
    return distances.min()

train_df['center_distance'] = train_df.apply(min_distance, axis=1)
train_df = train_df.drop(columns=['lat', 'lon'])


# Same for the test set
test_df['center_distance'] = test_df.apply(min_distance, axis=1)
test_df = test_df.drop(columns=['lat', 'lon'])

train_df.head()

,livingSpace,numberOfRooms,floor,rentGross,type_DUPLEX,type_FLAT,type_ROOF_FLAT,type_TERRACE_FLAT,type_other,hasGarage_True,hasGarage_unknown,hasParking_True,hasParking_unknown,hasBalcony_True,hasBalcony_unknown,hasElevator_True,hasElevator_unknown,center_distance
0,147.0,4.5,26.0,4950.0,0,1,0,0,0,0,1,0,1,1,0,1,0,0.015414
1,167.0,2.5,2.0,2225.0,0,0,0,0,1,1,0,0,1,0,1,0,1,0.006818
2,57.0,2.5,3.0,1782.0,0,1,0,0,0,1,0,0,1,1,0,0,1,0.013772
3,103.0,3.5,2.0,1600.0,0,1,0,0,0,0,1,1,0,1,0,0,1,0.131020
4,97.0,4.5,1.0,1985.0,0,1,0,0,0,0,1,1,0,0,1,1,0,0.048083


## 2.2 Average room size

A single new feature: living space divided by number of rooms, giving the average room size in $m^2$ per listing.

Two listings can share the same total `livingSpace` but split it very differently, for example $80m^2$ across 2 large rooms vs $80m^2$ across 4 small ones. That difference could likely affects `rentGross`, and neither `livingSpace` nor `numberOfRooms` alone captures it.

`livingSpace` and `numberOfRooms` are kept alongside the new ratio, not dropped, since each original column still carries information the ratio alone doesn't.

In [13]:
living_spaces = train_df['livingSpace'].values
number_of_rooms = train_df['numberOfRooms'].values
avg_room_size = living_spaces / number_of_rooms
train_df['averageRoomSize'] = avg_room_size

# Same for the test set
test_living_spaces = test_df['livingSpace'].values
test_number_of_rooms = test_df['numberOfRooms'].values
test_avg_room_size = test_living_spaces / test_number_of_rooms
test_df['averageRoomSize'] = test_avg_room_size

train_df.head()

,livingSpace,numberOfRooms,floor,rentGross,type_DUPLEX,type_FLAT,type_ROOF_FLAT,type_TERRACE_FLAT,type_other,hasGarage_True,hasGarage_unknown,hasParking_True,hasParking_unknown,hasBalcony_True,hasBalcony_unknown,hasElevator_True,hasElevator_unknown,center_distance,averageRoomSize
0,147.0,4.5,26.0,4950.0,0,1,0,0,0,0,1,0,1,1,0,1,0,0.015414,32.666667
1,167.0,2.5,2.0,2225.0,0,0,0,0,1,1,0,0,1,0,1,0,1,0.006818,66.800000
2,57.0,2.5,3.0,1782.0,0,1,0,0,0,1,0,0,1,1,0,0,1,0.013772,22.800000
3,103.0,3.5,2.0,1600.0,0,1,0,0,0,0,1,1,0,1,0,0,1,0.131020,29.428571
4,97.0,4.5,1.0,1985.0,0,1,0,0,0,0,1,1,0,0,1,1,0,0.048083,21.555556
